# 🚀 CineGrade AI - Free GPU Backend

Welcome! This notebook provides the 100% free NVIDIA T4 GPU backend required for CineGrade AI.

### Instructions (Quick Start Guide):
1. In the menu at the top, click **Runtime** ➔ **Run all** (or press Ctrl+F9).
2. A popup might warn you that this notebook was not authored by Google. Click **Run anyway**.
3. Scroll down to the bottom cell. It will print out a **Secure Localtunnel URL** (it looks like `https://xyz.loca.lt`).
4. Copy that URL and paste it into the CineGrade web app.

In [ ]:
# @title 1. Setup Environment (Please wait ~60 seconds for this to complete)
import os
from IPython.display import clear_output

print('⏳ Cloning repository and installing dependencies...')
if not os.path.exists('VideoColorGrading'):
    !git clone https://github.com/shameel0505/VideoColorGrading.git
%cd VideoColorGrading

!pip install -q diffusers==0.21.4 transformers==4.32.0 accelerate==0.22.0 omegaconf==2.3.0 einops==0.6.1 pillow_lut==1.1.0 decord==0.6.0 fastapi uvicorn python-multipart pydantic imageio_ffmpeg rawpy

!npm install -g localtunnel > /dev/null 2>&1

clear_output()
print('✅ Environment setup complete!')


In [ ]:
# @title 2. Start GPU Server & Generate URL
import subprocess
import time

print('⏳ Booting up FastAPI Server on port 8444...')
fastapi_proc = subprocess.Popen(['uvicorn', 'api:app', '--host', '127.0.0.1', '--port', '8444'])
time.sleep(3)

print('⏳ Establishing secure Localtunnel...')
tunnel_proc = subprocess.Popen(
    ['lt', '--port', '8444'],
    stdout=subprocess.PIPE, 
    stderr=subprocess.STDOUT,
    text=True
)

print('\n' + '='*60)
print('🎉 YOUR GPU SERVER IS STARTING!')
print('WAITING FOR URL... (This usually takes 5-10 seconds)')
print('='*60 + '\n')

url_found = False
for line in tunnel_proc.stdout:
    if 'your url is:' in line:
        full_url = line.strip().split('url is: ')[-1]
        print('\n' + '⭐'*30)
        print('\nCOPY THIS EXACT URL AND PASTE IT IN THE WEB APP:')
        print(f'\n--->  {full_url}  <---\n')
        print('⭐'*30 + '\n')
        print('⚠️ IMPORTANT: Click the URL above to open it in a new tab first!')
        print('⚠️ Click the "Click to Continue" button on the warning screen to whitelist your IP.')
        url_found = True
        break

if not url_found:
    print('❌ Failed to generate URL. Please restart the cell.')

try:
    tunnel_proc.wait()
except KeyboardInterrupt:
    fastapi_proc.terminate()
    print('\nServer stopped.')
